In [53]:
import torch
import torch.nn as nn
import random

chars = list("abcd ")
vocab = {ch: i for i, ch in enumerate(chars)} # Cada letra, ganha um número
inv_vocab = {i: ch for ch, i in vocab.items()}# Tabela de decodificação
vocab_size = len(vocab)

def encode(s): # Codifica letras em números
    return torch.tensor([vocab[c] for c in s], dtype=torch.long)

def decode(t): # Decodifica números em letras
    return ''.join(inv_vocab[int(x)] for x in t)

def random_seq(n=5): # Cria novas sequências
    return ''.join(random.choice(chars[:-1]) for _ in range(n))

# Gerar dados
pairs = [(encode(s), encode(s[::-1])) for s in [random_seq() for _ in range(200000)]]

max_len = max(len(x) for x, _ in pairs) # pega maior sequência

def pad(x):  # Preenche conjunto de dados em pad no último índice
    return torch.cat([x, torch.tensor([vocab[' ']] * (max_len - len(x)))], dim=0)

inputs = torch.stack([pad(x) for x, _ in pairs])
targets = torch.stack([pad(y) for _, y in pairs])

train_ds = torch.utils.data.TensorDataset(inputs, targets)
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=128, shuffle=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [54]:
# Veja um par
print(pairs[1])

(tensor([3, 1, 3, 0, 0]), tensor([0, 0, 3, 1, 3]))


In [55]:
# Definição do modelo Seq2Seq com GRU
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_size)
        self.gru = nn.GRU(emb_size, hidden_size, batch_first=True)

    def forward(self, x):
        x = self.embed(x)
        _, h = self.gru(x)
        return h  # [1, B, H]

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_size)
        self.gru = nn.GRU(emb_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h):
        """
        x: tensor que indica a parte prévia correta
        h: tensor que indica o estado do encoder da parte prévia
        """
        x = self.embed(x)
        out, h = self.gru(x, h)
        logits = self.fc(out)
        return logits, h # retorna o estado latente para atualizar o estado

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt):
        h = self.encoder(src)
        # usa contexto correto anterior e estado atual para prever o tgt[:, -1]
        logits, _ = self.decoder(tgt[:, :-1], h)
        return logits

In [56]:
# Código para usar o modelo treinado: inferência
def decode_step(decoder, token, h):
    logits, h = decoder(token, h) # obtém logits e atualiza estado da sequência
    next_token = logits[:, -1, :].argmax(-1, keepdim=True)
    return next_token, h

def predict(model, seq, max_len=10):
    model.eval()
    with torch.no_grad():
        src = pad(encode(seq)).unsqueeze(0).to(device, dtype=torch.long)
        h = model.encoder(src) # Obtém estado do modelo após processar entrada inicial

        # 'token' representa a geração passo a passo da sequência invertida
        token = torch.tensor([[vocab[' ']]], dtype=torch.long, device=device)
        seq_invertida = []
        for _ in range(max_len):
            token, h = decode_step(model.decoder, token, h)
            seq_invertida.append(token.item())
        return decode(seq_invertida)

In [57]:
# Preparação para treino
emb_size = 128
hidden_size = 128
encoder = Encoder(vocab_size, emb_size, hidden_size)
decoder = Decoder(vocab_size, emb_size, hidden_size)
model = Seq2Seq(encoder, decoder).to(device)

loss_fn = nn.CrossEntropyLoss(ignore_index=vocab[' ']) # ignora o pad: " "
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

In [58]:
# Execução do treino
for epoch in range(10):
    model.train()
    total_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device, dtype=torch.long), yb.to(device, dtype=torch.long)
        opt.zero_grad()
        logits = model(xb, yb)
        loss = loss_fn(logits.reshape(-1, vocab_size), yb[:, 1:].reshape(-1))
        loss.backward()
        opt.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: loss={total_loss/len(train_dl):.4f}")

Epoch 1: loss=0.0416
Epoch 2: loss=0.0001
Epoch 3: loss=0.0000
Epoch 4: loss=0.0000
Epoch 5: loss=0.0000
Epoch 6: loss=0.0000
Epoch 7: loss=0.0000
Epoch 8: loss=0.0000
Epoch 9: loss=0.0000
Epoch 10: loss=0.0000


In [59]:
# Test
count = 0
total_letras_corretas = 0
total_letras_avaliadas = 0
for _ in range(10):
    s = random_seq()
    alvo = s[::-1]
    pred = predict(model, s, max_len=len(s))
    if s == pred[::-1]:
        count += 1
    letras_corretas = sum(1 for char_alvo, char_pred in zip(alvo, pred) if char_alvo == char_pred)
    print(f"{s} -> {pred} : Acertou {letras_corretas} letras")
    total_letras_corretas += letras_corretas
    total_letras_avaliadas += len(alvo)
media_letras = total_letras_corretas / 10
print(f"Media de acertos: {media_letras}")
print(f"total de acertos: {count}")




daddd -> dddac : Acertou 4 letras
cddbc -> cbddc : Acertou 5 letras
bbbbb -> bbbbd : Acertou 4 letras
acbbc -> cbbca : Acertou 5 letras
aadcc -> ccdad : Acertou 4 letras
adaab -> baadc : Acertou 4 letras
aacbd -> dcbab : Acertou 2 letras
daaba -> abaad : Acertou 5 letras
aaaaa -> aaaac : Acertou 4 letras
badad -> dadca : Acertou 3 letras
Media de acertos: 4.0
total de acertos: 3


In [59]:
# Medias de desempenho para 3 execuções
# voc_size = 2000000
# emb_size = 64
# hidden_size = 64
# epochs = 10
# test = 10
# encoder - decoder
# GRU - GRU = 3,5; 3,9; 3,6
# RNN -  GRU = 1,5; 3,3; 2,8
# GRU - RNN = 2,6; 2,4; 2,9
# RNN - RNN = 1,8; 1,6; 1,3
# LSTM - LSTM = 3,4; 3,1; 3,1